## Script for Comparing Incorrectly Labeled Epochs

In [1]:
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import pickle
from sklearn.model_selection import train_test_split, KFold
import matplotlib
matplotlib.use('QtAgg') 

In [2]:
# importing model 
with open("M2_all.pkl", "rb") as f:
    model = pickle.load(f)

In [3]:
# importing features dataframe 
features_all = pd.read_pickle("training_features_19032026.pkl")

In [ ]:
X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

y_zygo = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all["Num_Contractions_Corr"].astype(int).to_numpy()

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate((y_zygo,y_corr),axis=0)       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [5]:
n1 = len(X_zygo)

idx_test_1 = idx_test[idx_test < n1]
idx_test_2 = idx_test[idx_test >= n1] - n1

In [6]:
X_test_zygo = X[idx_test_1]
X_test_corr = X[idx_test_2]
X_test = X[idx_test]

# get testing results 
y_pred_test_zygo = model.predict(X_test_zygo)   
y_pred_test_corr = model.predict(X_test_corr)   
y_pred_test  = model.predict(X_test)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   
y_pred_test = np.argmax(y_pred_test,axis=1)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step


In [7]:
subjects = features_all["Subject"] 
subject_subset = np.concatenate([subjects[idx_test_1],subjects[idx_test_2]])

epochs = features_all["Triggers_Order_Nap"] 
epochs_subset = np.concatenate([epochs[idx_test_1],epochs[idx_test_2]])

naps = features_all["Nap Number"] 
naps_subset = np.concatenate([naps[idx_test_1],naps[idx_test_2]])

In [9]:
print(np.shape(epoch_zygo_idx))
print(np.shape(epoch_corr_idx))
print(np.shape(epoch_idx))


(1, 80)
(1, 736)
(1, 220)


In [8]:
# get mismatched epoch indices where the tested indices dont equal predicted 
idx_zyg = np.shape(idx_test_1)[0]
idx_cor = np.shape(idx_test_2)[0] 

epoch_zygo_idx = np.where(y_zygo[idx_test_1] != y_pred_test_zygo)
epoch_corr_idx = np.where(y_corr[idx_test_2] != y_pred_test_corr)
epoch_idx = np.where(y[idx_test] != y_pred_test)

#mismatched_subj = subject_subset[epoch_idx]
#mismatched_epochs = epochs_subset[epoch_idx]
#mismatched_naps = naps_subset[epoch_idx]


#for i in range(len(mismatched_subj)):
    

 

Pre-Processing

In [ ]:
# making dataframe for epoch rescoring 
rescore_epochs = pd.DataFrame(
    columns=[
        "Subject", #HAVE
        "Nap Number", #HAVE
        "Triggers_Order_Nap", #HAVE
        "Num_Contractions_Zygo",
        "Num_Contractions_Corr",
        "Zygo",  
        "Corr",
        "Num_Contractions_Zygo_Pred",
        "Num_Contractions_Corr_Pred",
    ]
    )   


In [ ]:
print(np.shape(features_all["Zygo"]))
print(idx_test)

Scoring for Mismatched Epochs

In [ ]:
#GUI

### For now plotting ECG instead of corru
frq=250 # hardcode 250 
last_epoch_length = 5
current_index=0

def plot_figure(t):
    global df_triggers
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    if df_triggers.loc[t, 'Contraction_number'] != None and df_triggers.loc[t, 'Response_end_sample'] != None and df_triggers.loc[t, 'Response_start_sample'] != None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")
        title.set(**fdct)
    elif df_triggers.loc[t, 'Contraction_number'] ==0 and df_triggers.loc[t, 'Response_end_sample'] == None and df_triggers.loc[t, 'Response_start_sample'] == None:
        fdct = {'color': 'r'}
        title = fig.suptitle(f"Epoch {t + 1} - SCORED : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS")
        title.set(**fdct)
    else:
        title = fig.suptitle(f"Epoch {t + 1} - UNFINISHED SCORING : {df_triggers.loc[t, 'Contraction_number']} CONTRACTIONS; START = {df_triggers.loc[t, 'Response_start_sample']}; END = {df_triggers.loc[t, 'Response_end_sample']}")

    # Plot Corru (ECG for now)
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[0].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['ECG'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Chin", color="blue")
    else:    
        ax[0].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['ECG'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Chin", color="blue")
    #ax[0].set_ylim(-0.0002, 0.0002)
    ax[0].set_ylabel("ECG")

    # Plot Zygo
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[1].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['Zygo'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Zygo", color="black")
    else:   
        ax[1].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['Zygo'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Zygo", color="black")
    ax[1].set_ylim(-0.0002, 0.0002)
    ax[1].set_ylabel("Zygo")

    # Plot the trigger channel
    if t==239 or df_triggers['Stim_time_sample'][t+1]-df_triggers['Stim_time_sample'][t]>10*frq:
        ax[2].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq)), raw_wEEG_wZygo.get_data(picks=['Trigger'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t]+last_epoch_length*frq))[0], label="Trigger Channel", color="orange")
    else:
        ax[2].plot(range(int(df_triggers['Stim_time_sample'][t]-20),int(df_triggers['Stim_time_sample'][t+1]-1)), raw_wEEG_wZygo.get_data(picks=['Trigger'],start=int(df_triggers['Stim_time_sample'][t]-20),stop=int(df_triggers['Stim_time_sample'][t+1]-1))[0], label="Trigger Channel", color="orange")
    
    ax[2].set_xlabel("Stim_time_sample")
    ax[2].set_ylabel("Trigger")
    ax[2].set_ylim(0, 40)

    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.canvas.mpl_connect('button_press_event', on_click)

    plt.tight_layout()
    plt.show()

# Event handling function
def on_click(event):
    global current_index, fig, df_triggers
    if event.inaxes:  # Check if click occurred in any axes
        for i, a in enumerate(event.canvas.figure.axes):
            if event.inaxes == a:  # Check which subplot was clicked
                # print(f"Clicked in Subplot {i+1}")
                # print(f"Coordinates in data space: X = {event.xdata}, Y = {event.ydata}")
                if df_triggers['Muscle_type'][current_index]  == None:
                    df_triggers.loc[current_index, 'Response_start_sample' ] = int(event.xdata)
                    df_triggers.loc[current_index, 'RT_sec' ] = (df_triggers.loc[current_index, 'Response_start_sample' ] - df_triggers.loc[current_index, 'Stim_time_sample'])/frq
                    if i==0:
                        df_triggers.loc[current_index, 'Muscle_type'] = "Corru"
                    elif i==1:
                        df_triggers.loc[current_index, 'Muscle_type'] = "Zygo"           
                else:
                    df_triggers.loc[current_index, 'Response_end_sample' ] = int(event.xdata)

    if (df_triggers.loc[current_index, 'Contraction_number'] != None and df_triggers.loc[current_index, 'Response_end_sample'] != None and df_triggers.loc[current_index, 'Response_start_sample'] != None) or (df_triggers.loc[current_index, 'Contraction_number'] ==0):
        plt.close()  # Close current figure
        plot_figure(current_index)         
                    

# Keyboard press event handler
def on_key(event):
    global current_index, fig, df_triggers
    allowed_keys = {'0','1', '2', '3', '4', '5','5', '7', '8', '9'}
    # print(f"Key pressed: {event.key}")  # Print the key pressed
    key = event.key

    if key in allowed_keys:
        df_triggers.loc[current_index, 'Contraction_number'] = int(key) 
        if (df_triggers.loc[current_index, 'Response_end_sample'] != None and df_triggers.loc[current_index, 'Response_start_sample'] != None) or (df_triggers.loc[current_index, 'Contraction_number'] ==0):
            plt.close()  # Close current figure
            plot_figure(current_index) 
        
    elif event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(df_triggers)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(df_triggers) # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'r':  # Move to previous figure
        df_triggers.loc[current_index, 'Response_start_sample'] = None
        df_triggers.loc[current_index, 'Response_end_sample'] = None 
        df_triggers.loc[current_index, 'Contraction_number'] = None
        df_triggers.loc[current_index, 'Muscle_type'] = None
        df_triggers.loc[current_index, 'RT_sec'] = None
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'q':  # Custom action for specific key
        print("Quitting the plot!")
        plt.close()  # Close the figure
        df_triggers.to_csv(path_metadata+'/subject_{}_{}_metadata.csv'.format(subject,task_type),index=False)
        

# Plot the first figure
plot_figure(current_index)